# Caracal GRPO cyber (Colab T4) - RL direto CVE->CWE

**Por que Colab**: a quota de GPU do Colab e SEPARADA do Kaggle. Quando o Kaggle
esgota (30h/semana), esta roda igual.

**Runtime**: Menu -> Ambiente de execucao -> Alterar tipo -> **T4 GPU**.

**O que faz**: GRPO direto (sem outer loop RSI) partindo do Qwen2.5-Coder-3B
base, movendo o CVE->CWE de ~44% rumo ao Foundation-Sec-8B (72-75). Avalia a
cada bloco de 20 steps -> curva de aprendizado. Salva adapter + curva no Google
Drive, entao se o Colab cortar a sessao (~4h no free) e so re-rodar tudo com
RESUME=True que ele continua de onde parou.

**Regua pra ler o resultado**: base = step 0 do proprio run. Piso de colapso
(responder sempre CWE-79) = 0.273. Acima da base e progresso; perto de 0.273 com
unparsed baixo e colapso na classe majoritaria.

In [ ]:
# 1) Confirma a GPU (aborta se nao for GPU - GRPO em CPU nao termina nunca)
import torch, subprocess
assert torch.cuda.is_available(), 'Sem GPU: Ambiente de execucao -> T4 GPU'
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv']).decode())

In [ ]:
# 2) Drive pra persistir checkpoints atraves do corte de sessao do Colab
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/caracal_grpo_cyber'
os.makedirs(OUT, exist_ok=True)
RESUME = os.path.exists(OUT + '/curve.json')   # True se ja rodou antes
print('OUT =', OUT, '| RESUME =', RESUME)

In [ ]:
# 3) Stack pinado (mesma receita que roda em T4 SM 7.5, sem Unsloth)
!pip -q install 'torch==2.6.0' 'torchvision==0.21.0' --index-url https://download.pytorch.org/whl/cu124
!pip -q install 'transformers==4.49.0' 'peft==0.14.0' 'trl==0.15.2' 'accelerate>=1.0.0' 'datasets>=3.0.0' 'antlr4-python3-runtime==4.11'
!pip -q uninstall -y torchao 2>/dev/null
import transformers, torch
print('transformers', transformers.__version__, '| torch', torch.__version__)

In [ ]:
# 4) Codigo (com todos os fixes de hoje) + dataset CVE->CWE
import os
if not os.path.exists('/content/caracal-1'):
    !git clone --depth 1 -b s07-hybrid-agentic https://github.com/iterate-labs-ai/caracal-1.git /content/caracal-1
os.chdir('/content/caracal-1')
!git rev-parse --short HEAD
# arvore CWE local (evita o download da MITRE no hot path do reward)
import subprocess
subprocess.run(['python','-m','data.ignite.build_cyber_rcm'], check=True)
print('dataset pronto')

In [ ]:
# 5) GRPO direto. 120 steps em blocos de 20 (eval entre blocos = curva).
#    ~200s/step -> ~7h; o RESUME cobre o corte de sessao do Colab.
import sys, subprocess
cmd = [sys.executable, '-m', 'train.ignite.B_grpo_straight',
       '--base', 'Qwen/Qwen2.5-Coder-3B-Instruct',
       '--out', OUT, '--total-steps', '120', '--block', '20',
       '--lr', '1e-6', '--rank', '32', '--eval-n', '150']
if RESUME:
    cmd.append('--resume')
env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
subprocess.run(cmd, check=True, env=env)

In [ ]:
# 6) Curva final
import json
curve = json.load(open(OUT + '/curve.json'))
print('=== CURVA (base=step0 | piso de colapso CWE-79=0.273) ===')
for s, v in sorted(curve.items(), key=lambda x: int(x[0])):
    print(f"  step {int(s):3d}: acc={v['accuracy']:.3f} hier={v['hier']:.3f} unparsed={v['unparsed']:.2f}")
base = curve['0']['accuracy']
best = max(v['accuracy'] for v in curve.values())
print(f"\nbase {base:.3f} -> melhor {best:.3f}  (delta {best-base:+.3f})")